# Principal Component Analysis (PCA)

**Autor:** Guía generada automáticamente

**Fecha:** 2026-06-11

**Tags:** reducción de dimensionalidad, PCA, análisis exploratorio, visualización

**Propósito:** Introducir el concepto de PCA y mostrar una implementación práctica en Python con `scikit-learn`.

## Descripción

Principal Component Analysis (PCA) es una técnica de reducción de dimensionalidad que transforma un conjunto de variables correlacionadas en un conjunto de variables no correlacionadas llamadas componentes principales. Esta técnica es útil para explorar la estructura de los datos, visualizar conjuntos de datos multidimensionales y reducir el ruido antes de aplicar otros algoritmos de machine learning.

PCA busca proyecciones lineales de los datos que retengan la mayor cantidad de varianza posible. Técnicamente, esta reducción se puede obtener mediante la descomposición espectral de la matriz de covarianza o mediante la descomposición en valores singulares (SVD) de la matriz de datos centrados. En un contexto práctico, es frecuente usar PCA como paso previo a clasificación, clustering o visualización de datos.

## Objetivo del modelo

- Visualización de datos de alta dimensión en 2D o 3D.
- Preprocesamiento antes de clasificación o clustering.
- Compresión de datos y reducción de ruido.
- Análisis exploratorio en bioinformática, finanzas, visión por computador y más.

## Modelo matemático / fórmulas

Sea $X \in \mathbb{R}^{n \times p}$ la matriz de datos centrados (cada fila es una muestra y cada columna es una característica). PCA proyecta los datos sobre vectores ortogonales que capturan la mayor cantidad de varianza.

### 1. Descomposición espectral 

Un enfoque clásico es calcular la matriz de covarianza:

$$S = \frac{1}{n-1} \, X^\top X. $$

La descomposición espectral de $S$ es:

$$S = W \Lambda W^\top,$$

con $W = [w_1, w_2, \dots, w_p]$ la matriz de vectores propios ortogonales y $\Lambda = \mathrm{diag}(\lambda_1, \lambda_2, \dots, \lambda_p)$ los valores propios ordenados de mayor a menor. Las componentes principales se obtienen tomando los primeros $k$ autovectores:

$$Z = X W_k,$$

donde $W_k \in \mathbb{R}^{p \times k}$ contiene los $k$ vectores propios con mayor varianza.

**Consideraciones:**

- Requiere construir la matriz de covarianza $S$, que es de tamaño $p \times p$. 
- Es especialmente natural cuando $p$ es pequeño comparado con $n$. 
- Debe usarse datos centrados (media cero) y, en general, escalados si las características tienen unidades diferentes.

### 2. Descomposición en valores singulares (SVD)

Una alternativa más estable numéricamente es aplicar SVD directamente sobre la matriz centrada $X$:

$$X = U \Sigma V^\top,$$

donde $U \in \mathbb{R}^{n \times n}$, $\Sigma \in \mathbb{R}^{n \times p}$ y $V \in \mathbb{R}^{p \times p}$. Las columnas de $V$ son los vectores singulares derechos, y las primeras $k$ columnas definen la proyección:

$$Z = X V_k,$$

con $V_k \in \mathbb{R}^{p \times k}$. Los valores singulares $\sigma_i$ se relacionan con los valores propios de $S$ mediante $\lambda_i = \sigma_i^2 / (n-1)$.

**Consideraciones:**

- SVD es preferido cuando $n$ y $p$ tienen tamaños comparables o cuando $p > n$. 
- No requiere calcular la matriz de covarianza explícita. 
- Es más robusto frente a matrices de datos rectangulares y puede ser más eficiente en muchos casos.

En ambos enfoques, el objetivo es encontrar una base ortogonal que permita representar los datos originales con menos dimensiones, conservando la mayor parte de su varianza.

## Ventajas y desventajas

**Ventajas**
- Reduce dimensionalidad manteniendo la mayor parte de la varianza.
- Ayuda a visualizar datos multidimensionales.
- Mejora la eficiencia y puede reducir ruido.

**Desventajas**
- Es un método lineal; no captura relaciones no lineales.
- Las componentes no son fácilmente interpretables en términos de las variables originales.
- Requiere escalado previo cuando las variables están en diferentes unidades.

In [ ]:
# Importar librerías y configuración
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid')

## Datos de ejemplo

Usamos el conjunto de datos `iris` para ejemplificar PCA. Aunque es un conjunto pequeño, permite visualizar bien cómo funciona la reducción de dimensionalidad.

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = X.copy()
df['target'] = y
df['species'] = df['target'].map(dict(enumerate(target_names)))

df.head()

In [ ]:
# Estadísticas básicas
df.describe().T

## Preprocesamiento

PCA requiere que las variables tengan escala comparable. Escalamos las características antes de aplicar la descomposición.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2, random_state=RANDOM_SEED))
])

X_pca = pipeline.fit_transform(X)
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['species'] = df['species']
df_pca.head()

## Visualización de componentes principales

Representamos las dos primeras componentes principales para observar cómo los datos se agrupan en el espacio reducido.

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='species', palette='deep', s=80)
plt.title('Proyección PCA del conjunto Iris')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.legend(title='Especie')
plt.show()

In [ ]:
explained_variance = pipeline.named_steps['pca'].explained_variance_ratio_
explained_variance

## Interpretación de resultados

La proporción de varianza explicada por cada componente principal indica cuánto de la variabilidad total de los datos captura cada dimensión reducida. En este ejemplo, las dos primeras componentes suelen retener gran parte de la varianza del conjunto Iris.

## Conclusiones y siguientes pasos

- PCA es útil para reducir dimensionalidad y visualizar datos cuando existe correlación entre variables.
- Requiere escalado previo para obtener resultados adecuados.
- Para datos no lineales, conviene explorar métodos como Kernel PCA o t-SNE.

Siguientes pasos recomendados:
- Evaluar el número de componentes con la varianza acumulada.
- Combinar PCA con modelos supervisados para comparar desempeño.
- Explorar PCA en conjuntos de datos más grandes y heterogéneos.

## Reproducibilidad

Ejecuta el siguiente comando para instalar las dependencias necesarias:

```bash
pip install numpy pandas scikit-learn matplotlib seaborn jupyter
```